<a href="https://colab.research.google.com/github/abdu1lah69/churn-assessment/blob/main/Assessment_1/customer_churn_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import urllib.request
# no need to upload the dataset :)
DATASET_PATH = "customer_churn.csv"
DATASET_URL = "https://raw.githubusercontent.com/abdu1lah69/churn-assessment/main/Assessment_1/customer_churn.csv"

if not os.path.exists(DATASET_PATH):
    print("Downloading customer_churn.csv from abdu1lah69 repository...")
    urllib.request.urlretrieve(DATASET_URL, DATASET_PATH)
    print("Downloaded successfully! Ready to execute pipeline.")
else:
    print("Dataset customer_churn.csv is already available! Ready to execute pipeline.")


In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# Custom transformer to cap outliers using the IQR method
# This dependency is used by the modular preprocessing pipeline
class TrainFittedIQRCapper(BaseEstimator, TransformerMixin):

    def __init__(self, factor=1.5):
        self.factor = factor
        self.bounds_ = {}

    # IQR method
    def fit(self, X, y=None):
        X_arr = np.asarray(X, dtype=float)
        self.bounds_ = {}
        for col_idx in range(X_arr.shape[1]):
            col_vals = X_arr[:, col_idx]
            valid_vals = col_vals[~np.isnan(col_vals)]
            if len(valid_vals) > 0:
                q1 = np.percentile(valid_vals, 25)
                q3 = np.percentile(valid_vals, 75)
                iqr = q3 - q1
                lower = q1 - self.factor * iqr
                upper = q3 + self.factor * iqr
                self.bounds_[col_idx] = (lower, upper)
            else:
                self.bounds_[col_idx] = (-np.inf, np.inf)
        return self

    # cliping b/w bounds calculated
    def transform(self, X):
        X_arr = np.asarray(X, dtype=float).copy()
        for col_idx, (lower, upper) in self.bounds_.items():
            if col_idx < X_arr.shape[1]:
                X_arr[:, col_idx] = np.clip(X_arr[:, col_idx], lower, upper)
        return X_arr

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return np.asarray([f"x{i}" for i in range(len(self.bounds_))], dtype=object)

In [ ]:
#Step 1 ---> dataset integrity
import hashlib
import pandas as pd
import os

DATA_PATH = "customer_churn.csv"
OUTPUT_DIR = "outputs"
RANDOM_STATE = 42
EXPECTED_HASH = "16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Integrity check
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(f"Dataset file not found at: {DATA_PATH}")

hasher = hashlib.sha256()
with open(DATA_PATH, "rb") as file:
    for chunk in iter(lambda: file.read(65536), b""):
        hasher.update(chunk)
computed_hash = hasher.hexdigest()

print(f"Data Checksum (SHA-256): {computed_hash}")
if computed_hash.lower() == EXPECTED_HASH.lower():
    print("Integrity Check: PASS (Checksum matches reference dataset).")
else:
    print(f"Integrity Check: MISMATCH (Expected {EXPECTED_HASH}, got {computed_hash}).")

# Loading the dataset
df_raw = pd.read_csv(DATA_PATH)
display(df_raw.head())

In [ ]:
#Step 2 ---> PreProcessing
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

df = df_raw.copy()

# Droping unrequired cols.
if "customerID" in df.columns:
    df.drop(columns=["customerID"], inplace=True)
    print("Dropped identifier column: 'customerID'.")

# Blank values handling
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
nan_count = df["TotalCharges"].isna().sum()
print(f"Detected {nan_count} blank entries in TotalCharges coerced to NaN.")

# Map target variable Churn to 1/0
if df["Churn"].dtype == object or str(df["Churn"].dtype).startswith("str"):
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Graph to show distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=df["Churn"], hue=df["Churn"], palette=["blue", "red"], legend=False)
plt.title("Customer Churn Distribution (0 = Retained, 1 = Churned)")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "churn_distribution.png"), dpi=300)
plt.show()

# spliting
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f" Split created: Training Set = {len(X_train)} Rows, Test Set = {len(X_test)} Rows.")

In [ ]:
#Step 3 ---> Cleaning and Processing Training Data
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
#Cols distribution
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
passthrough_cols = ["SeniorCitizen"] if "SeniorCitizen" in X_train.columns else []
cat_cols = [
    c for c in X_train.columns
    if c not in num_cols and c not in passthrough_cols and (
        X_train[c].dtype == "object" or isinstance(X_train[c].dtype, (pd.CategoricalDtype, pd.StringDtype))
    )
]

# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("capper", TrainFittedIQRCapper(factor=1.5)),
    ("scaler", StandardScaler()),
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
])

# Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
    ] + ([("pass", "passthrough", passthrough_cols)] if passthrough_cols else []),
    remainder="drop",
)

# Transform data
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print(f"Preprocessing complete.")

In [ ]:
#Step 4 ---> Unsupervised Learning K-Means and PCA
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import numpy as np

k_range = range(2, 7)
inertias = []
silhouette_scores = []

print("Evaluating best cluster number in range 2-7 :")
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_train_proc)
    score = silhouette_score(X_train_proc, labels)
    inertias.append(km.inertia_)
    silhouette_scores.append(score)
    print(f"  k={k}: Inertia = {km.inertia_:.2f}, Silhouette Score = {score:.4f}")

# Selection best
best_idx = int(np.argmax(silhouette_scores))
selected_k = list(k_range)[best_idx]
print(f"Selected k = {selected_k}")


final_kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = final_kmeans.fit_predict(X_train_proc)

# PCA Projection
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_train_proc)
explained_var = pca.explained_variance_ratio_

# Plotting clusters
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="viridis", alpha=0.6)
plt.title(f"Unsupervised Customer Clusters (k={selected_k}) via PCA")
plt.xlabel(f"PC1 ({explained_var[0] * 100:.1f}% variance)")
plt.ylabel(f"PC2 ({explained_var[1] * 100:.1f}% variance)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_customer_clusters.png"), dpi=300)
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(list(k_range), inertias, marker="o")
ax1.set(xlabel="Clusters", ylabel="Inertia", title="K-Means Elbow")
ax2.plot(list(k_range), silhouette_scores, marker="o")
ax2.set(xlabel="Clusters", ylabel="Silhouette score", title="Cluster Separation")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "kmeans_evaluation.png"), dpi=300)
plt.close(fig)

In [ ]:
#Step 5 ---> Supervised Learning LogisticRegression and RandomForest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train_proc, y_train)
print("Logistic Regression model training completed.")

# 2. Random Forest Hyperparameter Grid Search
rf_base = RandomForestClassifier(random_state=RANDOM_STATE)
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [6, 10, None],
    "min_samples_split": [2, 5],
}

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
)
grid_search.fit(X_train_proc, y_train)
best_rf_model = grid_search.best_estimator_
print(f"Random Forest search completed. Best parameters found: {grid_search.best_params_}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": lr_model,
    "Random Forest": best_rf_model
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plt.figure(figsize=(7, 5))

metrics_summary = {}
for idx, (name, clf) in enumerate(models.items()):
    # Predictions
    preds = clf.predict(X_test_proc)
    probs = clf.predict_proba(X_test_proc)[:, 1]

    # Benchmarking stats
    metrics_summary[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1 Score": f1_score(y_test, preds, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, probs)
    }

    print(f"\n--- {name} Classification Report ---")
    print(classification_report(y_test, preds, zero_division=0))

    # Confusion Matrix heatmaps
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx], cbar=False)
    axes[idx].set_title(f"Confusion Matrix: {name}")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("True")

    # ROC Curve mapping
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {metrics_summary[name]['ROC AUC']:.3f})")

# Show confusion matrices
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=300)

# Configure ROC chart
plt.plot([0, 1], [0, 1], "k--")
plt.title("ROC Curves comparison (Held-out Test)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_curves.png"), dpi=300)
plt.show()

# Benchmark Summary Table
summary_df = pd.DataFrame(metrics_summary).T
display(summary_df.round(4))

import json

feature_names = preprocessor.get_feature_names_out()
importance = best_rf_model.feature_importances_
top = np.argsort(importance)[-10:]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(feature_names[top], importance[top])
ax.set(xlabel="Importance", title="Random Forest Feature Importances")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "feature_importance.png"), dpi=300)
plt.close(fig)

evaluation = {
    "supervised_models": metrics_summary,
    "unsupervised_models": {
        "selected_k": selected_k,
        "selection_rationale": "Highest silhouette score among tested cluster counts.",
        "selected_silhouette_score": float(silhouette_scores[best_idx]),
        "pca_explained_variance_ratio": explained_var.tolist(),
    },
}
with open(os.path.join(OUTPUT_DIR, "evaluation_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(evaluation, f, indent=2)


In [ ]:
# The monolithic class execution has been deprecated to rely entirely on the modular section steps above.

In [ ]:
# Step 3: Display Actual Evaluation Metrics
import json
import pandas as pd

metrics_file = os.path.join(OUTPUT_DIR, "evaluation_metrics.json")
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        metrics_data = json.load(f)
    print("=" * 60)
    print("SUPERVISED MODEL PERFORMANCE ON HELD-OUT TEST SET")
    print("=" * 60)
    df_sup = pd.DataFrame(metrics_data["supervised_models"]).T
    try:
        display(df_sup.round(4))
    except NameError:
        print(df_sup.round(4))

    print("\n" + "=" * 60)
    print("UNSUPERVISED CLUSTERING PERFORMANCE")
    print("=" * 60)
    unsup = metrics_data["unsupervised_models"]
    print(f"Selected k: {unsup['selected_k']}")
    print(f"Selection Rationale: {unsup['selection_rationale']}")
    print(f"Silhouette Score: {unsup['selected_silhouette_score']:.4f}")
    print(f"PCA Explained Variance: PC1={unsup['pca_explained_variance_ratio'][0]*100:.1f}%, PC2={unsup['pca_explained_variance_ratio'][1]*100:.1f}%")
else:
    print("outputs/evaluation_metrics.json not found. Run the evaluation cell first.")


In [ ]:
# Step 4: Display Generated Publication-Grade Visualizations
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_files = [
    ("churn_distribution.png", "Class Distribution (Imbalance Visualization)"),
    ("kmeans_evaluation.png", "K-Means Inertia (Elbow) and Silhouette Analysis"),
    ("pca_customer_clusters.png", "PCA 2D Customer Behavioral Segmentation"),
    ("confusion_matrices.png", "Confusion Matrices (Logistic Regression vs Random Forest)"),
    ("roc_curves.png", "Receiver Operating Characteristic (ROC) Comparison"),
    ("feature_importance.png", "Random Forest Feature Importances (MDI)")
]

for pf, title in plot_files:
    path = os.path.join(OUTPUT_DIR, pf)
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(title, fontsize=13, fontweight="bold", pad=12)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Image not found: {path}")
